## SELECCIÓN DE MODELO DE MACHINE LEARNING

### By:
Cristian David Ceballos Velez

### Date:
2026-08-25

### Description:
Selección manual del mejor modelo de machine learning para predecir `disease` en el dataset
`corazon.csv`, comparando contra el modelo base (heurístico) de la etapa anterior. Se reutiliza
el pipeline de preprocesamiento de `4-feat_eng` y se sigue la estructura de referencia:
[Selección de Modelo - Jose R. Zapata](https://joserzapata.github.io/post/ciencia-datos-proyecto-python/6-model_selection/)

**Métrica principal: Recall** (definida en la etapa de modelo base), por el alto costo clínico
de los Falsos Negativos en diagnóstico de enfermedad cardíaca.

**Objetivo:** el modelo seleccionado debe superar el Recall del modelo base heurístico
(ver `5-models/5modelobase_cdcv_20260825.ipynb`).

## 📚 1. Importar librerías

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import dump, load
from scipy.stats import f_oneway
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    ShuffleSplit,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

print("Pandas version:", pd.__version__)

## 💾 2. Cargar datos

In [ ]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

df = pd.read_parquet(DATA_DIR / "02_intermediate" / "corazon_type_fixed.parquet", engine="pyarrow")

TARGET = "disease"
df[TARGET] = df[TARGET].astype(int)

# Convertir tipos nullable a estándar (mismo ajuste que en 4-feat_eng)
cols_int_a_float = ["age", "rest_bp", "chol", "max_hr", "ca"]
for col in cols_int_a_float:
    df[col] = df[col].astype("float64")

cols_bool_a_float = ["fbs", "exang"]
for col in cols_bool_a_float:
    df[col] = df[col].astype("float64")

df = df.drop_duplicates()
df.info()

## 👷 3. Preparación de datos

In [ ]:
selected_features = [c for c in df.columns]  # se usan todas las columnas disponibles

heart_df = df[selected_features].copy()
heart_df.isna().sum()

## 👨‍🏭 4. Pipeline de preprocesamiento

Se reutiliza la misma estructura de `4-feat_eng`: numéricas (imputación + escalado) y categóricas nominales (imputación + One-Hot).

In [ ]:
cols_numeric = ["age", "rest_bp", "chol", "max_hr", "old_peak", "slope", "ca", "fbs", "exang"]
cols_categoric = ["sex", "chest_pain", "rest_ecg", "thal"]

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, cols_numeric),
        ("categoric", categorical_pipe, cols_categoric),
    ]
)
preprocessor

## ✂️ 5. Train / Test split

In [ ]:
X_features = heart_df.drop(columns=[TARGET])
Y_target = heart_df[TARGET]

x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, test_size=0.2, stratify=Y_target, random_state=42
)

print("Train:", x_train.shape, y_train.shape)
print("Test:", x_test.shape, y_test.shape)

## 🤖 6. Modelos candidatos

Se evalúan 8 modelos de al menos 6 familias distintas (lineales, basados en árboles, ensambles,
vecinos cercanos, probabilísticos, margen máximo):

- Regresión Logística
- Análisis Discriminante Lineal (LDA)
- Descenso de Gradiente Estocástico (SGD)
- Máquina de Vectores de Soporte (SVC)
- K-Vecinos más Cercanos (KNN)
- Naive Bayes Gaussiano
- Árbol de Decisión
- Random Forest
- Gradient Boosting

In [ ]:
models = {
    "logistic": LogisticRegression(solver="liblinear", random_state=42),
    "lda": LinearDiscriminantAnalysis(),
    "sgd": SGDClassifier(random_state=42),
    "svc": SVC(random_state=42),
    "knn": KNeighborsClassifier(),
    "naive_bayes": GaussianNB(),
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "random_forest": RandomForestClassifier(random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}
len(models)

## 🥇 7. Primera selección de modelos

Entrenamiento simple con hiperparámetros por defecto, para descartar los modelos con peor desempeño o con sobreajuste evidente antes de invertir tiempo en validación cruzada.

In [ ]:
def summarize_classification(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc": roc_auc_score(y_true, y_pred),
    }


def build_model(classifier, preprocessor: ColumnTransformer, x_tr, y_tr, x_te, y_te) -> dict:
    """Entrena un modelo dentro de un pipeline y calcula métricas en train y test."""
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", classifier)])
    pipe.fit(x_tr, y_tr)

    y_pred_train = pipe.predict(x_tr)
    y_pred_test = pipe.predict(x_te)

    return {
        "train": summarize_classification(y_tr, y_pred_train),
        "test": summarize_classification(y_te, y_pred_test),
    }

In [ ]:
result_dict = {}
for model_name, model in models.items():
    result_dict[model_name] = build_model(model, preprocessor, x_train, y_train, x_test, y_test)

result_dict

### 7.1 Comparación de modelos (train vs. test)

In [ ]:
metrics = ["accuracy", "precision", "recall", "f1", "roc"]
model_names = list(result_dict.keys())

data_train = {m: {mod: result_dict[mod]["train"][m] for mod in model_names} for m in metrics}
data_test = {m: {mod: result_dict[mod]["test"][m] for mod in model_names} for m in metrics}

df_train = pd.DataFrame(data_train)
df_test = pd.DataFrame(data_test)

for metric in metrics:
    fig, ax = plt.subplots(figsize=(10, 4))
    width = 0.35
    df_train[metric].plot(
        kind="bar", ax=ax, width=width, position=1, label="Train", color="steelblue"
    )
    df_test[metric].plot(kind="bar", ax=ax, width=width, position=0, label="Test", color="orange")
    ax.axhline(df_train[metric].mean(), color="steelblue", linestyle="--", linewidth=1)
    ax.axhline(df_test[metric].mean(), color="orange", linestyle="--", linewidth=1)
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"Comparación de {metric.capitalize()} - Train vs Test")
    ax.set_xticklabels(model_names, rotation=45, ha="right")
    ax.legend()
    plt.tight_layout()
    plt.show()

### 7.2 Detectar sobreajuste y bajo desempeño

In [ ]:
df_combined = pd.concat([df_train.add_suffix("_train"), df_test.add_suffix("_test")], axis=1)

for metric in metrics:
    df_combined[f"{metric}_diff"] = df_combined[f"{metric}_train"] - df_combined[f"{metric}_test"]

overfitting_threshold = 0.15
overfitting_models = df_combined[
    (df_combined["accuracy_diff"] > overfitting_threshold)
    | (df_combined["precision_diff"] > overfitting_threshold)
    | (df_combined["recall_diff"] > overfitting_threshold)
    | (df_combined["f1_diff"] > overfitting_threshold)
    | (df_combined["roc_diff"] > overfitting_threshold)
]

mean_train = df_combined[[f"{m}_train" for m in metrics]].mean()
mean_test = df_combined[[f"{m}_test" for m in metrics]].mean()

low_performance_models = df_combined[
    (df_combined["accuracy_train"] < mean_train["accuracy_train"])
    & (df_combined["accuracy_test"] < mean_test["accuracy_test"])
    & (df_combined["recall_train"] < mean_train["recall_train"])
    & (df_combined["recall_test"] < mean_test["recall_test"])
    & (df_combined["f1_train"] < mean_train["f1_train"])
    & (df_combined["f1_test"] < mean_test["f1_test"])
]

print("Modelos con posible sobreajuste (train-test > 0.15):", list(overfitting_models.index))
print(
    "Modelos con desempeño por debajo del promedio (accuracy, recall y f1):",
    list(low_performance_models.index),
)

### 7.3 Modelos descartados

Con base en la sección anterior, elimina del diccionario `selected_models` los modelos con
**bajo desempeño** (por debajo del promedio en las métricas clave). Los modelos con sobreajuste
no se descartan automáticamente aquí — se controlan con regularización/hiperparámetros en el
tuning; solo se descartan si además tienen bajo desempeño en test.

**Ajusta esta lista según el resultado real de la celda anterior:**

In [ ]:
modelos_descartados = []  # <-- AJUSTAR con los nombres que salieron en low_performance_models

selected_models = {name: model for name, model in models.items() if name not in modelos_descartados}
print("Modelos que continúan a validación cruzada:", list(selected_models.keys()))

## 🔁 8. Validación cruzada de los modelos seleccionados

In [ ]:
scoring_metrics = ["accuracy", "f1", "precision", "recall"]
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

cv_results = {metric: {} for metric in scoring_metrics}

for model_name, model in selected_models.items():
    model_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    for metric in scoring_metrics:
        cv_results[metric][model_name] = cross_val_score(
            model_pipe, x_train, y_train, cv=kfold, scoring=metric
        )

cv_results_df = {metric: pd.DataFrame(cv_results[metric]) for metric in scoring_metrics}

In [ ]:
mean_std_rows = []
for metric_name in scoring_metrics:
    for model_name in selected_models:
        scores = cv_results_df[metric_name][model_name]
        mean_std_rows.append(
            {"Model": model_name, "Metric": metric_name, "Mean": scores.mean(), "Std": scores.std()}
        )

mean_std_df = pd.DataFrame(mean_std_rows)
mean_std_df.pivot(index="Model", columns="Metric", values="Mean").round(4)

In [ ]:
for metric_name in scoring_metrics:
    plt.figure(figsize=(10, 5))
    cv_results_df[metric_name].boxplot()
    plt.title(f"Validación Cruzada - {metric_name.capitalize()}")
    plt.ylabel(metric_name.capitalize())
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 🎯 9. Comparación estadística (Recall)

In [ ]:
result_recall_df = cv_results_df["recall"]
result_recall_df

In [ ]:
statistic, p_value = f_oneway(*[result_recall_df[col] for col in result_recall_df.columns])

print(f"Estadístico ANOVA: {statistic:.4f}")
print(f"p-valor: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print("Hay diferencia estadísticamente significativa entre los modelos (Recall).")
else:
    print("NO hay diferencia estadísticamente significativa entre los modelos (Recall).")

**Interpretación:** si el ANOVA no muestra diferencia significativa, la elección de qué
modelos pasar a la fase de tuning puede basarse en criterios secundarios (interpretabilidad,
tiempo de entrenamiento, capacidad de mejora con hiperparámetros). Selecciona 2 o 3 modelos para
la siguiente etapa — se recomienda que sean de familias distintas, por ejemplo un modelo lineal
(Regresión Logística o LDA) y un modelo basado en árboles/ensambles (Random Forest o
Gradient Boosting), para comparar enfoques diferentes.

## 🛠️ 10. Optimización de hiperparámetros

### 10.1 Regresión Logística

In [ ]:
parameters_logistic = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.1, 0.4, 0.8, 1, 2, 5],
}

logistic_pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(solver="liblinear", random_state=42)),
    ]
)

grid_logistic = GridSearchCV(
    logistic_pipe, parameters_logistic, cv=5, scoring="recall", return_train_score=True
)
grid_logistic.fit(x_train, y_train)

print("Mejores hiperparámetros:", grid_logistic.best_params_)
print("Mejor Recall (CV):", grid_logistic.best_score_)

### 10.2 Random Forest

In [ ]:
parameters_rf = {
    "model__max_depth": [4, 5, 7, 9, 10],
    "model__max_features": [2, 3, 4, 5, 6],
    "model__criterion": ["gini", "entropy"],
}

rf_pipe = Pipeline(
    steps=[("preprocessor", preprocessor), ("model", RandomForestClassifier(random_state=42))]
)

grid_rf = GridSearchCV(rf_pipe, parameters_rf, cv=5, scoring="recall", return_train_score=True)
grid_rf.fit(x_train, y_train)

print("Mejores hiperparámetros:", grid_rf.best_params_)
print("Mejor Recall (CV):", grid_rf.best_score_)

### 10.3 Gradient Boosting

In [ ]:
parameters_gb = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [2, 3, 4],
    "model__learning_rate": [0.01, 0.1, 0.2],
}

gb_pipe = Pipeline(
    steps=[("preprocessor", preprocessor), ("model", GradientBoostingClassifier(random_state=42))]
)

grid_gb = GridSearchCV(gb_pipe, parameters_gb, cv=5, scoring="recall", return_train_score=True)
grid_gb.fit(x_train, y_train)

print("Mejores hiperparámetros:", grid_gb.best_params_)
print("Mejor Recall (CV):", grid_gb.best_score_)

## 📏 11. Evaluación final en test set

In [ ]:
logistic_model = grid_logistic.best_estimator_
rf_model = grid_rf.best_estimator_
gb_model = grid_gb.best_estimator_

candidatos_finales = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
}

for nombre, modelo in candidatos_finales.items():
    y_pred = modelo.predict(x_test)
    print(f"\n=== {nombre} ===")
    print(classification_report(y_test, y_pred))

In [ ]:
for nombre, modelo in candidatos_finales.items():
    y_pred = modelo.predict(x_test)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(f"Matriz de confusión - {nombre}")
    plt.show()

## 🥇 12. Comparación final y selección del mejor modelo

In [ ]:
ax = plt.gca()
roc_plots = {}
for nombre, modelo in candidatos_finales.items():
    roc_plots[nombre] = RocCurveDisplay.from_estimator(
        modelo, x_test, y_test, ax=ax, alpha=0.8, name=nombre
    )
plt.title("Curva ROC - Comparación de modelos finales")
plt.show()

**Selección del modelo final:** compara el Recall en test entre los tres candidatos y frente
al modelo base heurístico de la etapa anterior. **Documenta aquí cuál seleccionaste y por qué**
(ej. mejor Recall, mejor balance con Precision, menor tiempo de entrenamiento, mayor
interpretabilidad). Deja el modelo elegido asignado a la variable `modelo_final` para las
siguientes secciones.

In [ ]:
modelo_final = rf_model  # <-- AJUSTAR según los resultados reales obtenidos
nombre_modelo_final = "Random Forest"  # <-- AJUSTAR

## ⚠️ 13. Chequeo de underfitting / overfitting

Si la brecha train-test del modelo final es grande (sobreajuste) o el desempeño en ambos conjuntos es bajo (subajuste), hay que volver a los pasos anteriores (otros modelos/hiperparámetros) o incluso a la etapa de feature engineering.

In [ ]:
y_pred_train_final = modelo_final.predict(x_train)
y_pred_test_final = modelo_final.predict(x_test)

metrics_train = summarize_classification(y_train, y_pred_train_final)
metrics_test = summarize_classification(y_test, y_pred_test_final)

comparacion = pd.DataFrame({"Train": metrics_train, "Test": metrics_test})
comparacion["Diferencia"] = comparacion["Train"] - comparacion["Test"]
comparacion

**Diagnóstico** (completar con base en la tabla anterior):
- Diferencia grande (> 0.15) en Recall entre train y test → señal de **overfitting**: considerar
  regularización adicional, reducir complejidad del modelo, o conseguir más datos.
- Desempeño bajo en ambos (train y test) → señal de **underfitting**: considerar un modelo más
  complejo, más features, o revisar la calidad del feature engineering.
- Diferencia pequeña y desempeño aceptable → el modelo generaliza razonablemente bien,
  se puede continuar con el análisis de aprendizaje (siguiente sección).

## 📈 14. Learning Curve (Recall) del modelo final

In [ ]:
common_params = {
    "X": x_train,
    "y": y_train,
    "train_sizes": np.linspace(0.1, 1.0, 5),
    "cv": ShuffleSplit(n_splits=50, test_size=0.2, random_state=123),
    "n_jobs": -1,
    "return_times": True,
}

scoring_metric = "recall"

train_sizes, train_scores, test_scores, fit_times, score_times = learning_curve(
    modelo_final, **common_params, scoring=scoring_metric
)

train_mean, train_std = np.mean(train_scores, axis=1), np.std(train_scores, axis=1)
test_mean, test_std = np.mean(test_scores, axis=1), np.std(test_scores, axis=1)
fit_times_mean, fit_times_std = np.mean(fit_times, axis=1), np.std(fit_times, axis=1)
score_times_mean, score_times_std = np.mean(score_times, axis=1), np.std(score_times, axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, train_mean, "o-", label="Training score")
ax.plot(train_sizes, test_mean, "o-", color="orange", label="Cross-validation score")
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.3)
ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.3, color="orange")
ax.set_title(f"Learning Curve - {nombre_modelo_final} (Recall)")
ax.set_xlabel("Ejemplos de entrenamiento")
ax.set_ylabel(scoring_metric)
ax.legend(loc="best")
plt.show()

print("Training Sizes:", train_sizes)
print("Training Scores Mean:", train_mean)
print("Test Scores Mean:", test_mean)

## ⚙️ 15. Escalabilidad (tiempo de entrenamiento y score)

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 10), sharex=True)

ax[0].plot(train_sizes, fit_times_mean, "o-")
ax[0].fill_between(
    train_sizes, fit_times_mean - fit_times_std, fit_times_mean + fit_times_std, alpha=0.3
)
ax[0].set_ylabel("Fit time (s)")
ax[0].set_title(f"Escalabilidad - {nombre_modelo_final}")

ax[1].plot(train_sizes, score_times_mean, "o-")
ax[1].fill_between(
    train_sizes, score_times_mean - score_times_std, score_times_mean + score_times_std, alpha=0.3
)
ax[1].set_ylabel("Score time (s)")
ax[1].set_xlabel("Número de ejemplos de entrenamiento")

plt.show()

print("Fit Times Mean:", fit_times_mean)
print("Score Times Mean:", score_times_mean)

## 💾 16. Guardar el modelo final (pipeline completo)

In [ ]:
MODELS_DIR = Path.cwd().resolve().parents[1] / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

nombre_archivo = (
    MODELS_DIR / "corazon_classification-random_forest-v1.joblib"
)  # <-- AJUSTAR según modelo_final
dump(modelo_final, nombre_archivo)
print("Modelo guardado en:", nombre_archivo)

### 16.1 Verificación del modelo guardado

In [ ]:
my_model = load(nombre_archivo)

print(x_test.head())
print("Predicciones:", my_model.predict(x_test.head()))

## 📊 17. Análisis de resultados

_Completar con los valores numéricos reales obtenidos:_

- **Modelo final seleccionado:** (nombre) con hiperparámetros: (completar desde `grid_search.best_params_`)
- **Recall en test:** (valor) vs. **Recall del modelo base heurístico:** (valor de la etapa anterior)
  → ¿en cuánto mejora el modelo de ML al baseline?
- **Variables más importantes:** si el modelo final es basado en árboles (Random Forest,
  Gradient Boosting), extraer `modelo_final.named_steps["model"].feature_importances_` junto con
  `preprocessor.get_feature_names_out()` para identificar qué variables más influyen en la
  predicción.
- **Comportamiento de la Learning Curve:** ¿el modelo se beneficiaría de más datos, o ya
  convergió?
- **Escalabilidad:** ¿el tiempo de entrenamiento crece linealmente con el tamaño de los datos?
  ¿es aceptable para un eventual reentrenamiento periódico?

In [ ]:
# Feature importance (solo si el modelo final es basado en árboles)
if hasattr(modelo_final.named_steps["model"], "feature_importances_"):
    feature_names = modelo_final.named_steps["preprocessor"].get_feature_names_out()
    importances = modelo_final.named_steps["model"].feature_importances_

    importancia_df = pd.DataFrame(
        {"feature": feature_names, "importancia": importances}
    ).sort_values("importancia", ascending=False)

    importancia_df.plot(kind="barh", x="feature", y="importancia", figsize=(8, 6), legend=False)
    plt.title(f"Importancia de variables - {nombre_modelo_final}")
    plt.xlabel("Importancia")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    importancia_df  # noqa: B018


## 🧑‍🔬 18. Recomendaciones

1. **Modelo recomendado para producción:** (completar) — justificar con base en Recall,
   generalización (learning curve) y escalabilidad.
2. **Variables a priorizar/recolectar:** con base en la importancia de variables, priorizar la
   calidad de captura de esas columnas en producción (menos nulos, validación más estricta).
3. **Umbral de decisión:** dado que Recall es prioritario, evaluar si ajustar el umbral de
   clasificación (por debajo de 0.5) mejora el Recall sin sacrificar demasiada Precision —
   relevante para la etapa de interpretación de modelos.
4. **Monitoreo:** una vez en producción, monitorear el Recall en datos nuevos para detectar
   *drift* del modelo.

## 💡 19. Propuestas e ideas

1. **Modelos adicionales**: evaluar XGBoost o LightGBM, que suelen superar a Gradient Boosting
   de scikit-learn en velocidad y desempeño en datasets tabulares medianos.
2. **Búsqueda de hiperparámetros más amplia**: usar `RandomizedSearchCV` o `Optuna` en vez de
   `GridSearchCV` para explorar más combinaciones con el mismo presupuesto de cómputo.
3. **Balanceo de clases**: aunque el dataset está razonablemente balanceado, probar técnicas
   como `class_weight="balanced"` o SMOTE si se prioriza aún más el Recall.
4. **Interpretabilidad**: usar SHAP o `permutation_importance` en la siguiente etapa
   (`7-interpretation`) para validar y explicar mejor las predicciones del modelo final.
5. **Ensamble de modelos**: combinar Random Forest y Gradient Boosting en un `VotingClassifier`
   o `StackingClassifier` para potencialmente mejorar el Recall.